In [17]:
#!pip install boto3
#!pip install awscli
#!pip install farm-haystack

In [8]:
# Upload a file to an S3 bucket
import boto3
from botocore.exceptions import NoCredentialsError

def upload_to_s3(file_name, bucket_name, object_name=None):
    s3_client = boto3.client('s3')
    try:
        s3_client.upload_file(file_name, bucket_name, object_name or file_name)
        print(f"File {file_name} uploaded to {bucket_name}.")
        return True
    except NoCredentialsError:
        print("Credentials not available.")
        return False

In [6]:
# hashing function for dublicate checks
import hashlib

def calculate_hash(file_path):
    sha256_hash = hashlib.sha256()
    with open(file_path, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()


In [7]:
# checker function for dublicates in s3 bucket
def is_duplicate(file_hash, existing_hashes):
    return file_hash in existing_hashes

def get_existing_hashes(bucket_name):
    s3 = boto3.client('s3')
    hashes = []
    for obj in s3.list_objects_v2(Bucket=bucket_name).get('Contents', []):
        hashes.append(obj['Key'])
    return hashes


In [10]:
# workflow function
def upload_with_duplicate_check(file_path, bucket_name):
    file_hash = calculate_hash(file_path)
    existing_hashes = get_existing_hashes(bucket_name)
    
    if is_duplicate(file_hash, existing_hashes):
        print("Duplicate file detected. Not uploading.")
        return False
    
    # If not duplicate, upload file and save hash to S3
    success = upload_to_s3(file_path, bucket_name, file_hash)
    if success:
        print("File uploaded successfully.")
    return success


In [21]:
# pulling documents from s3 with haystack
from haystack.document_stores import FAISSDocumentStore
from haystack.utils import clean_wiki_text, convert_files_to_docs, fetch_archive_from_http
from haystack.pipelines import DocumentSearchPipeline
from haystack.nodes import DensePassageRetriever, FARMReader, TransformersReader

document_store = FAISSDocumentStore(faiss_index_factory_str="Flat")


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------------------- -------------------- 0.8/1.6 MB 4.8 MB/s eta 0:00:01
   -------------------------------- ------- 1.3/1.6 MB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 3.0 MB/s eta 0:00:00
Failed to build faiss-cpu


  error: subprocess-exited-with-error
  
  × Building wheel for faiss-cpu (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [8 lines of output]
      running bdist_wheel
      running build
      running build_py
      running build_ext
      building 'faiss._swigfaiss' extension
      swigging faiss\faiss\python\swigfaiss.i to faiss\faiss\python\swigfaiss_wrap.cpp
      swig.exe -python -c++ -Doverride= -I/usr/local/include -Ifaiss -doxygen -DSWIGWIN -o faiss\faiss\python\swigfaiss_wrap.cpp faiss\faiss\python\swigfaiss.i
      error: command 'swig.exe' failed: None
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for faiss-cpu
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (faiss-cpu)


ImportError: Failed to import 'faiss'. Run 'pip install farm-haystack[faiss]'. Original error: No module named 'faiss'

In [ ]:
# ingesting documents to haystack 
from haystack.document_stores import S3DocumentStore

def ingest_to_rag_system(bucket_name, file_hash):
    document_store = S3DocumentStore(bucket_name=bucket_name)
    # Assuming the file is a text or PDF that can be ingested
    documents = convert_files_to_dicts(dir_path=f"s3://{bucket_name}/{file_hash}")
    document_store.write_documents(documents)
